# ***Spamify*** - A ML Model to detect spam email messages

**Import necessary libraries**

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


**1. Loading The Dataset**

In [40]:
df=pd.read_csv('/content/spam.csv', encoding='latin-1')
df = df.iloc[:, :2] # Keep only the first two columns (Category, Message)
df.columns = ['Category', 'Message'] # Rename columns for clarity


In [41]:
df.shape

(5572, 2)

**2. Drop the duplicates**

In [42]:
df.drop_duplicates(inplace=True)

In [43]:
print(df.shape)

(5157, 2)


**3. Check for any null values in the dataset**

In [44]:
df.isnull().sum()

,0
Category,0
Message,0


In [45]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


**4. Change wherever ham is appearing and change it into spam**

In [46]:
df["Category"] = df["Category"].replace({"ham": "Not spam"})

**5. Train the model and evaluate model score**

In [47]:
# Re-run data splitting
msg=df['Message']
cat=df['Category']

# Re-run train-test split with corrected 'cat'
# Adding an explicit check for the presence of both categories before stratify
if len(cat.value_counts()) < 2:
    print('Warning: Only one category present after replacement. Cannot stratify.')
    (msg_train,msg_test,cat_train,cat_test)=train_test_split(msg,cat,test_size=0.2,random_state=1)
else:
    (msg_train,msg_test,cat_train,cat_test)=train_test_split(msg,cat,test_size=0.2,random_state=1,stratify=cat)

# Re-initialize and re-fit CountVectorizer as data might have changed distribution
cv=CountVectorizer(stop_words="english")
features = cv.fit_transform(msg_train)

# Re-initialize and re-train the model
model=MultinomialNB()
model.fit(features,cat_train)

# Re-transform test data and evaluate the model
features_test=cv.transform(msg_test)
print(f"New model score: {model.score(features_test,cat_test)}")


New model score: 0.9903100775193798


**6. Check predictions of the model**

In [48]:
predictions = model.predict(features_test)

# Create a DataFrame to display actual vs. predicted for a sample
prediction_df = pd.DataFrame({
    'Message': msg_test.reset_index(drop=True),
    'Actual Category': cat_test.reset_index(drop=True),
    'Predicted Category': predictions
})

print("Sample of Model Predictions:")
display(prediction_df.head(10))

Sample of Model Predictions:


,Message,Actual Category,Predicted Category
0,You unbelievable faglord,Not spam,Not spam
1,Horrible gal. Me in sch doing some stuff. How ...,Not spam,Not spam
2,Probably gonna swing by in a wee bit,Not spam,Not spam
3,FREE entry into our Â£250 weekly comp just sen...,spam,spam
4,I think u have the wrong number.,Not spam,Not spam
5,Watching tv now. I got new job :),Not spam,Not spam
6,Doc prescribed me morphine cause the other pai...,Not spam,Not spam
7,"Er, hello, things didnât quite go to plan â...",Not spam,Not spam
8,Finish liao... U?,Not spam,Not spam
9,Free Top ringtone -sub to weekly ringtone-get ...,spam,spam


In [49]:
from sklearn.metrics import confusion_matrix, classification_report

# Generate the confusion matrix
cm = confusion_matrix(cat_test, predictions)
print('Confusion Matrix:')
display(pd.DataFrame(cm, index=model.classes_, columns=model.classes_))

# Generate the classification report
report = classification_report(cat_test, predictions)
print('\nClassification Report:')
print(report)

Confusion Matrix:


,Not spam,spam
Not spam,898,6
spam,4,124



Classification Report:
              precision    recall  f1-score   support

    Not spam       1.00      0.99      0.99       904
        spam       0.95      0.97      0.96       128

    accuracy                           0.99      1032
   macro avg       0.97      0.98      0.98      1032
weighted avg       0.99      0.99      0.99      1032



# **Conclusion: The model achieved a significant 99% accuracy**